# 3. Sistema de Recomendacao de Livros

## IAA012 - Frameworks de IA
### Especializacao em Inteligencia Artificial Aplicada - UFPR/SEPT

---

Implementacao de um **Sistema de Recomendacao de Livros** utilizando **Filtragem Colaborativa** com **Redes Neurais** e **Embeddings**, seguindo a arquitetura apresentada na aula **FRA - Aula 22 - 4.3 Resolucao do Exercicio de Sistemas de Recomendacao**.

**Arquitetura do modelo:**
- Camada de Embedding para usuarios
- Camada de Embedding para livros
- Concatenacao dos vetores de embedding
- Camada Densa oculta (1024 neuronios, ativacao ReLU)
- Camada de saida (1 neuronio, linear) para predicao da nota

**Base de dados:** `Base_livros.csv`
- **Colunas:** ISBN, Titulo, Autor, Ano, Editora, ID_usuario, Notas

## 1. Importacao das Bibliotecas

In [ ]:
# Compatibilidade TensorFlow/Keras
import os
os.environ["TF_USE_LEGACY_KERAS"] = "1"

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Embedding, Flatten, Dense, Concatenate
from tensorflow.keras.optimizers import SGD
from sklearn.model_selection import train_test_split

import warnings
warnings.filterwarnings('ignore')

print(f"TensorFlow version: {tf.__version__}")
print(f"GPU disponivel: {tf.config.list_physical_devices('GPU')}")

## 2. Carregamento e Exploracao dos Dados

In [ ]:
# Carregar a base de dados
df = pd.read_csv('Base_livros.csv')

print("=" * 60)
print("INFORMACOES DA BASE DE DADOS")
print("=" * 60)
print(f"Total de registros:  {df.shape[0]:,}")
print(f"Colunas:             {list(df.columns)}")
print(f"Usuarios unicos:     {df['ID_usuario'].nunique():,}")
print(f"Livros unicos (ISBN): {df['ISBN'].nunique():,}")
print(f"Faixa de notas:      {df['Notas'].min()} a {df['Notas'].max()}")
print(f"Media das notas:     {df['Notas'].mean():.2f}")
print()
df.head()

## 3. Pre-processamento dos Dados

### Codificacao Categorica

Os IDs de usuarios e ISBNs sao convertidos em indices inteiros sequenciais (0, 1, 2, ...) para serem usados como entrada nas camadas de Embedding.

In [ ]:
# Codificacao dos IDs de usuarios e livros em indices sequenciais
df['ID_usuario'] = pd.Categorical(df['ID_usuario'])
df['user_id'] = df['ID_usuario'].cat.codes

df['ISBN'] = pd.Categorical(df['ISBN'])
df['book_id'] = df['ISBN'].cat.codes

n_users = df['user_id'].nunique()
n_books = df['book_id'].nunique()

print(f"Usuarios (codificados): {n_users:,}")
print(f"Livros (codificados):   {n_books:,}")

# Criar mapeamentos para uso posterior nas recomendacoes
book_info = df.drop_duplicates('book_id').set_index('book_id')[['ISBN', 'Titulo', 'Autor']].to_dict('index')
user_to_code = dict(zip(df['ID_usuario'], df['user_id']))
user_rated_books = df.groupby('user_id')['book_id'].apply(set).to_dict()

In [ ]:
# Preparar arrays de entrada e saida
user_input = df['user_id'].values
book_input = df['book_id'].values
ratings = df['Notas'].values.astype(np.float32)

# Centralizar notas pela media do treino (conforme arquitetura da aula)
ratings_mean = float(ratings.mean())
ratings_centered = ratings - ratings_mean

print(f"Media das notas: {ratings_mean:.2f}")
print(f"Notas centralizadas: media = {ratings_centered.mean():.4f}")

In [ ]:
# Divisao treino/teste (80/20)
indices = np.arange(len(ratings))
train_idx, test_idx = train_test_split(indices, test_size=0.2, random_state=42)

u_train, u_test = user_input[train_idx], user_input[test_idx]
b_train, b_test = book_input[train_idx], book_input[test_idx]
r_train, r_test = ratings_centered[train_idx], ratings_centered[test_idx]

print(f"Conjunto de treino:    {len(r_train):,} registros")
print(f"Conjunto de teste:     {len(r_test):,} registros")

## 4. Construcao do Modelo (Arquitetura Aula 22)

A arquitetura segue o modelo apresentado na **Aula 22 - 4.3**:

1. **Embedding de Usuario:** Mapeia cada usuario para um vetor denso de dimensao fixa
2. **Embedding de Livro:** Mapeia cada livro (ISBN) para um vetor denso de dimensao fixa
3. **Concatenacao:** Junta os dois vetores de embedding
4. **Camada Densa (1024, ReLU):** Aprende interacoes nao-lineares entre usuario e livro
5. **Saida (1, Linear):** Prediz a nota centralizada

**Otimizador:** SGD com learning_rate=0.08 e momentum=0.9
**Loss:** MSE (Erro Quadratico Medio)
**Epochs:** 25

In [ ]:
# Hiperparametros (conforme Aula 22)
EMBEDDING_DIM = 10
LEARNING_RATE = 0.08
MOMENTUM = 0.9
BATCH_SIZE = 1024
EPOCHS = 25

In [ ]:
# Construcao do modelo com Functional API do Keras
# Entrada do usuario
user_inp = Input(shape=(1,), name='user_input')
user_emb = Embedding(n_users, EMBEDDING_DIM, name='user_embedding')(user_inp)
user_vec = Flatten()(user_emb)

# Entrada do livro
book_inp = Input(shape=(1,), name='book_input')
book_emb = Embedding(n_books, EMBEDDING_DIM, name='book_embedding')(book_inp)
book_vec = Flatten()(book_emb)

# Concatenacao dos embeddings + Camada densa
concat = Concatenate()([user_vec, book_vec])
dense = Dense(1024, activation='relu')(concat)

# Saida linear (regressao)
output = Dense(1, name='rating_output')(dense)

# Criar e compilar o modelo
model = Model(inputs=[user_inp, book_inp], outputs=output)
model.compile(
    loss='mse',
    optimizer=SGD(learning_rate=LEARNING_RATE, momentum=MOMENTUM)
)

model.summary()

## 5. Treinamento do Modelo (epoch = 25)

In [ ]:
print(f"Treinando modelo: {EPOCHS} epochs, batch_size={BATCH_SIZE}")
print(f"Otimizador: SGD(lr={LEARNING_RATE}, momentum={MOMENTUM})")
print(f"Loss: MSE (Erro Quadratico Medio)")
print()

In [ ]:
# Treinamento do modelo
history = model.fit(
    [u_train, b_train], r_train,
    batch_size=BATCH_SIZE,
    epochs=EPOCHS,
    validation_data=([u_test, b_test], r_test),
    verbose=2
)

## 6. Graficos de Avaliacao do Modelo (Loss)

O grafico de **funcao de perda (loss)** e fundamental para avaliar o comportamento do modelo durante o treinamento. Ele mostra como o erro evolui ao longo das epocas tanto no conjunto de treino quanto no de validacao.

In [ ]:
# Grafico de Loss (Treino vs Validacao)
plt.figure(figsize=(10, 6))
plt.plot(history.history['loss'], 'b-', label='loss (Treino)', linewidth=2)
plt.plot(history.history['val_loss'], 'r-', label='val_loss (Validacao)', linewidth=2)
plt.xlabel('Epocas')
plt.ylabel('Erro Quadratico Medio (MSE)')
plt.title('Funcao de Perda (Loss)')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# Metricas finais
print("\n" + "=" * 60)
print("METRICAS FINAIS DE LOSS")
print("=" * 60)
print(f"Loss final (treino):     {history.history['loss'][-1]:.4f}")
print(f"Loss final (validacao):  {history.history['val_loss'][-1]:.4f}")
print(f"Loss inicial (treino):   {history.history['loss'][0]:.4f}")
print(f"Reducao total:           {history.history['loss'][0] - history.history['loss'][-1]:.4f}")

### Explicacao dos Graficos de Loss

**O que e a funcao de perda (Loss)?**

A funcao de perda mede o quao distantes as predicoes do modelo estao dos valores reais. Neste caso, usamos o **MSE (Mean Squared Error)** — a media dos erros ao quadrado entre a nota prevista e a nota real dada pelo usuario.

**O que observamos no grafico:**

1. **Queda inicial rapida (primeiras epocas):** O modelo aprende rapidamente os padroes mais evidentes dos dados — por exemplo, que certos usuarios tendem a dar notas altas e certos livros sao geralmente bem avaliados. Esta e a fase de maior aprendizado.

2. **Estabilizacao gradual (epocas intermediarias/finais):** Apos as primeiras epocas, a curva de loss se achata progressivamente, indicando que o modelo ja capturou os padroes principais e agora faz ajustes incrementais nos pesos.

3. **Relacao entre curvas de treino e validacao:**
   - **Curvas proximas:** Indicam boa **generalizacao** — o modelo performa de forma similar em dados que ja viu (treino) e dados novos (validacao).
   - **Curva de treino muito abaixo da validacao:** Indica **overfitting** — o modelo esta memorizando os dados de treino em vez de aprender padroes gerais.
   - **Ambas as curvas altas e estagnadas:** Indica **underfitting** — o modelo e simples demais ou precisa de mais treinamento.

4. **Convergencia:** Quando ambas as curvas estabilizam em valores proximos, o modelo atingiu um ponto de convergencia, significando que mais epocas provavelmente nao trariam melhoria significativa.

**Conclusao:** O grafico de loss e a principal ferramenta para diagnosticar se o modelo esta aprendendo adequadamente e se ha necessidade de ajustes na arquitetura ou nos hiperparametros.

## 7. Sistema de Recomendacao de Livros

A funcao abaixo gera recomendacoes para um usuario especifico:
1. Identifica todos os livros que o usuario **ainda nao avaliou**
2. Prediz a nota que o usuario daria a cada livro nao avaliado
3. Retorna os livros com **maior nota prevista** (Top-N)

In [ ]:
def recomendar_livros(usuario_id, top_n=10):
    """
    Gera recomendacoes de livros para um usuario.
    
    Parametros:
    - usuario_id: ID original do usuario (ex: 276729)
    - top_n: numero de recomendacoes desejadas
    
    Retorna:
    - DataFrame com os livros recomendados e notas previstas
    """
    if usuario_id not in user_to_code:
        print(f"Usuario {usuario_id} nao encontrado!")
        return None
    
    user_code = user_to_code[usuario_id]
    
    # Livros ja avaliados pelo usuario
    livros_avaliados = user_rated_books.get(user_code, set())
    
    # Livros candidatos (nao avaliados)
    all_books = np.arange(n_books)
    livros_novos = np.array([b for b in all_books if b not in livros_avaliados])
    
    if len(livros_novos) == 0:
        print(f"Usuario {usuario_id} ja avaliou todos os livros!")
        return None
    
    # Predicao em lote
    users_array = np.full(len(livros_novos), user_code)
    preds = model.predict([users_array, livros_novos], batch_size=4096, verbose=0).flatten()
    
    # Desnormalizar (adicionar a media de volta) e limitar ao intervalo [0, 10]
    preds_original = preds + ratings_mean
    preds_original = np.clip(preds_original, 0, 10)  # Limita ao intervalo valido
    
    # Selecionar Top-N
    top_idx = np.argsort(preds_original)[::-1][:top_n]
    top_books = livros_novos[top_idx]
    top_notas = preds_original[top_idx]
    
    # Montar DataFrame com resultados
    resultados = []
    for book_code, nota in zip(top_books, top_notas):
        info = book_info.get(book_code, {})
        resultados.append({
            'ISBN': info.get('ISBN', 'N/A'),
            'Titulo': info.get('Titulo', 'N/A'),
            'Autor': info.get('Autor', 'N/A'),
            'Nota_Prevista': round(nota, 2)
        })
    
    return pd.DataFrame(resultados)

In [ ]:
def mostrar_historico(usuario_id, top_n=5):
    """Mostra os livros ja avaliados pelo usuario (melhores notas)."""
    if usuario_id not in user_to_code:
        print(f"Usuario {usuario_id} nao encontrado!")
        return None
    
    historico = df[df['ID_usuario'] == usuario_id][['Titulo', 'Autor', 'Notas']]
    return historico.sort_values('Notas', ascending=False).head(top_n)

## 8. Exemplo de Recomendacao para um Usuario

Vamos selecionar um usuario ativo da base e demonstrar o funcionamento do sistema de recomendacao, mostrando:
1. O **historico** de livros avaliados pelo usuario
2. As **recomendacoes** geradas pelo modelo (livros que o usuario ainda nao leu)

In [ ]:
# Encontrar usuarios mais ativos (com mais avaliacoes)
usuarios_ativos = df.groupby('ID_usuario').size().sort_values(ascending=False)
print("Top 5 usuarios mais ativos:")
print(usuarios_ativos.head())

# Selecionar o usuario mais ativo para demonstracao
USUARIO_EXEMPLO = usuarios_ativos.index[0]
print(f"\nUsuario selecionado para recomendacao: {USUARIO_EXEMPLO}")
print(f"Total de avaliacoes deste usuario: {usuarios_ativos[USUARIO_EXEMPLO]}")

In [ ]:
# Historico de avaliacoes do usuario
print(f"{'=' * 70}")
print(f"HISTORICO DO USUARIO {USUARIO_EXEMPLO}")
print(f"{'=' * 70}")
print("\nLivros com melhores notas dadas pelo usuario:")
display(mostrar_historico(USUARIO_EXEMPLO, top_n=5))

In [ ]:
# Gerar recomendacoes para o usuario
print(f"{'=' * 70}")
print(f"RECOMENDACOES PARA O USUARIO {USUARIO_EXEMPLO}")
print(f"{'=' * 70}")

recomendacoes = recomendar_livros(USUARIO_EXEMPLO, top_n=10)
print("\nTop 10 livros recomendados:")
display(recomendacoes)

In [ ]:
# Visualizacao grafica das recomendacoes
if recomendacoes is not None:
    plt.figure(figsize=(12, 6))
    
    # Truncar titulos longos para melhor visualizacao
    titulos = [t[:40] + '...' if len(str(t)) > 40 else str(t) for t in recomendacoes['Titulo']]
    notas = recomendacoes['Nota_Prevista'].values
    
    bars = plt.barh(range(len(titulos)), notas, color='steelblue', edgecolor='navy')
    plt.yticks(range(len(titulos)), titulos)
    plt.xlabel('Nota Prevista (escala 0-10)')
    plt.title(f'Top 10 Recomendacoes para Usuario {USUARIO_EXEMPLO}')
    plt.xlim(0, 10)  # Escala fixa de 0 a 10
    plt.gca().invert_yaxis()
    
    # Adicionar valores nas barras
    for bar, nota in zip(bars, notas):
        plt.text(bar.get_width() + 0.1, bar.get_y() + bar.get_height()/2,
                 f'{nota:.1f}', va='center', fontweight='bold')
    
    plt.tight_layout()
    plt.show()

### Explicacao das Recomendacoes

**Como o modelo gera as recomendacoes:**

O sistema utiliza **Filtragem Colaborativa com Redes Neurais**, que funciona da seguinte forma:

1. **Embeddings de Usuario:** Cada usuario e representado por um vetor numerico (embedding) que captura suas preferencias latentes — por exemplo, se gosta de ficcao cientifica, romance, etc. Essas preferencias nao sao explicitamente definidas; sao aprendidas automaticamente durante o treinamento.

2. **Embeddings de Livro:** Cada livro tambem e representado por um vetor numerico que captura suas caracteristicas latentes — como genero, estilo de escrita, complexidade, etc.

3. **Concatenacao + Camada Densa:** Os vetores de usuario e livro sao concatenados e passados por uma camada densa com 1024 neuronios (ativacao ReLU), que aprende **interacoes nao-lineares** entre as preferencias do usuario e as caracteristicas do livro.

4. **Predicao da Nota:** A camada de saida gera um unico valor — a nota prevista que o usuario daria ao livro.

**Por que funciona:**
- Usuarios com gostos similares possuem embeddings proximos no espaco vetorial
- Livros com caracteristicas similares tambem possuem embeddings proximos
- A rede neural aprende a "combinar" embeddings de usuarios e livros compativeis, prevendo notas altas para combinacoes que devem resultar em boa experiencia de leitura

**Interpretacao do ranking:**
- Os livros recomendados sao aqueles com **maior nota prevista** pelo modelo
- Quanto maior o score, maior a afinidade prevista entre o usuario e o livro
- O modelo so recomenda livros que o usuario **ainda nao avaliou**

In [ ]:
# Recomendacao para um segundo usuario (validacao)
print("=" * 70)
print("RECOMENDACAO PARA UM SEGUNDO USUARIO")
print("=" * 70)

outro_usuario = usuarios_ativos.index[5]
print(f"\nUsuario: {outro_usuario} ({usuarios_ativos[outro_usuario]} avaliacoes)")

print("\nHistorico (Top 5 melhores notas):")
display(mostrar_historico(outro_usuario, top_n=5))

print(f"\nTop 5 Recomendacoes:")
display(recomendar_livros(outro_usuario, top_n=5))

## 9. Avaliacao Final do Modelo

In [ ]:
# Avaliacao no conjunto de teste
test_loss = model.evaluate([u_test, b_test], r_test, verbose=0)

# Calcular MAE manualmente na escala original
preds_test = model.predict([u_test, b_test], batch_size=4096, verbose=0).flatten()
preds_original = preds_test + ratings_mean
preds_original = np.clip(preds_original, 0, 10)  # Limitar ao intervalo valido [0, 10]
reais_original = r_test + ratings_mean
mae_original = np.mean(np.abs(preds_original - reais_original))

print("=" * 60)
print("AVALIACAO FINAL DO MODELO")
print("=" * 60)
print(f"Loss (MSE) no teste:    {test_loss:.4f}")
print(f"MAE na escala 0-10:     {mae_original:.2f}")
print(f"\nInterpretacao: O modelo erra, em media, {mae_original:.2f} pontos")
print(f"na escala de notas de 0 a 10.")
print(f"\nNota: As predicoes sao limitadas ao intervalo [0, 10].")

In [ ]:
# Grafico: Predicoes vs Valores Reais (com predicoes limitadas a [0, 10])
plt.figure(figsize=(8, 6))
plt.scatter(reais_original, preds_original, alpha=0.1, s=5, color='steelblue')
plt.plot([0, 10], [0, 10], 'r--', linewidth=2, label='Predicao Perfeita')
plt.xlabel('Nota Real')
plt.ylabel('Nota Prevista (limitada a 0-10)')
plt.title('Predicoes vs Valores Reais')
plt.legend()
plt.grid(True, alpha=0.3)
plt.xlim(-0.5, 10.5)
plt.ylim(-0.5, 10.5)
plt.tight_layout()
plt.show()

## 10. Conclusao

### Resumo da Arquitetura (Aula 22 - 4.3)

| Componente | Configuracao |
|------------|-------------|
| Embedding de Usuario | Dimensao: 10 |
| Embedding de Livro | Dimensao: 10 |
| Concatenacao | user_vec + book_vec |
| Camada Densa | 1024 neuronios, ReLU |
| Saida | 1 neuronio, Linear |
| Otimizador | SGD (lr=0.08, momentum=0.9) |
| Loss | MSE |
| Epochs | 25 |
| Batch Size | 1024 |
| Normalizacao | Centralizacao pela media |

### Resultado

O modelo de filtragem colaborativa com embeddings consegue aprender padroes de preferencia dos usuarios a partir das avaliacoes da base `Base_livros.csv`, gerando recomendacoes personalizadas de livros. Os graficos de loss demonstram a convergencia do treinamento, e as recomendacoes geradas refletem a capacidade do modelo de identificar afinidades latentes entre usuarios e livros.

In [ ]:
print("Sistema de Recomendacao de Livros - Concluido!")
print(f"\nResumo do modelo:")
print(f"  Usuarios:        {n_users:,}")
print(f"  Livros:          {n_books:,}")
print(f"  Embedding dim:   {EMBEDDING_DIM}")
print(f"  Epochs:          {EPOCHS}")
print(f"  Otimizador:      SGD(lr={LEARNING_RATE}, momentum={MOMENTUM})")
print(f"  Loss final:      {history.history['loss'][-1]:.4f}")
print(f"  Val_loss final:  {history.history['val_loss'][-1]:.4f}")
print(f"  MAE (0-10):      {mae_original:.2f}")